## Try RAG with LLM on your custome database 

To use this notebook, run ```jupyter notebook``` from the notebook directory.
> ⚠️ **Warning:** You should run this command after setting up a development environment (-dev) as detailed  in the README file.

Install the necessary modules:

In [1]:
import os
import torch
from transformers import pipeline
from rag.utils import pretty_print, get_leaf_classes
from rag.retrieval import Retriever
from rag.config import Config as RAGConfig
from rag.models.embedding_models.embedding_models import EmbeddingModel

Set the config:

In [2]:
config = RAGConfig()

Available embedding models:

In [3]:
for cls_ in get_leaf_classes(EmbeddingModel):
    print(f" - {cls_.name}")

 - all-MiniLM-L6-v2


Set your inference parameters:

In [4]:
generic_prompt = "You are an assistant, short answer using the following information: "  # You can try your own prompt
model_name = "h2oai/h2o-danube3-500m-chat"  # Must be a valid HuggingFace model path (access token might be required)
embedding_model = "all-MiniLM-L6-v2"  # Must be among the embedding models previously listed
verbose = True

# To test the rag database of the repository
rag_db_path = os.path.join(os.path.dirname(os.getcwd()), "src", "data", "rag_database.pkl")
# To test the rag database generated using the notebook
# rag_db_path = os.path.join(os.getcwd(), "rag_database.pkl") 

Initialize the pipeline:

In [5]:
# define the LLM
pipe = pipeline(
    "text-generation",
    model=model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"\rLLM model used: {model_name}")

# RAG's retriever
rag_config = RAGConfig()
retriever = Retriever(config=rag_config,
                      embedding_model=embedding_model,
                      rag_db=rag_db_path)

Device set to use cuda:0


LLM model used: h2oai/h2o-danube3-500m-chat


To ask a question to the LLM with RAG.

Enter you question:

In [6]:
user_input = "Where can I inject insulin?"

Run:

In [7]:
# contextual information is retrieved based on the user query
chunk_list, similarity_list, metadata_list = retriever(query=user_input)

# Command detection
if "intent" in metadata_list[0]:
    pretty_print(name="Intent detected", result_dictionary={
    "Command n°": metadata_list[0]['intent'],
    "LLM is bypassed": True
    })
    exit()

rag_prompt = generic_prompt
if len(chunk_list) > 0:
    rag_prompt += ' '.join(chunk_list)
else:
    rag_prompt += "No information."
rag_prompt += '\n' + user_input

messages = [{"role": "user", "content": rag_prompt}]

prompt = pipe.tokenizer.apply_chat_template(messages,
                                            tokenize=False,
                                            add_generation_prompt=True)

res = pipe(prompt,
           return_full_text=False,
           max_new_tokens=256)

llm_output = res[0]["generated_text"]

# Print the retrieved results
pretty_print(name="LLM", result_dictionary={
    "Question": user_input,
    "Answer": llm_output,
})

╭─ LLM:
│  • Question: Where can I inject insulin?
│  • Answer: Insulin can be injected in various locations depending on the type of insulin and the individual's needs. Here are some common injection sites:

1. **Abdomen**: The abdomen is a convenient location for injecting insulin due to its size and ease of access. It is often used for short-term insulin therapy, such as for meals or snacks.

2. **Legs**: The legs are another common injection site for insulin. This is because they are relatively close to the body and can be easily accessed. However, it is important to note that the leg injection site may not be suitable for long-term insulin therapy, as the body may not absorb the insulin as effectively.

3. **Buttock**: The buttock is a less common injection site for insulin. It is often used for short-term insulin therapy, such as for meals or snacks.

4. **Other Locations**: In some cases, insulin may be injected into the thigh, upper arm, or upper arm. However, these locations a